# Using Trackio with Tune

<a id="try-anyscale-quickstart-tune-wandb" href="https://console.anyscale.com/register/ha?render_flow=ray&utm_source=ray_docs&utm_medium=docs&utm_campaign=tune-wandb">
    <img src="../../_static/img/run-on-anyscale.svg" alt="try-anyscale-quickstart">
</a>
<br></br>

(tune-trackio-ref)=

[Trackio](https://huggingface.co/docs/trackio/index) is a open-source, lightweight, free experiment tracking 
Python library built on top of Hugging Face Datasets and Spaces 🤗. It has a local first design and the experiments can be viewed with a Gradio dashboard locally or on the HF Hub

```{image} /images/trackio_logo_full.png
:align: center
:alt: Trackio
:height: 80px
:target: https://huggingface.co/docs/trackio/index/
```

Ray Tune currently offers two lightweight integrations for Weights & Biases.
One is the {ref}`TrackioLoggerCallback <air-trackio-logger>`, which automatically logs
metrics reported to Tune to the Wandb API.

The other one is the {ref}`setup_trackio() <air-trackio-setup>` function, which can be
used with the function API. It automatically
initializes the Wandb API with Tune's training information. You can just use the
Wandb API like you would normally do, e.g. using `wandb.log()` to log your training
process.

```{contents}
:backlinks: none
:local: true
```

## Running A Trackio Example

In the following example we're going to use both of the above methods, namely the `TrackioLoggerCallback` and
the `setup_trackio` function to log metrics.

As the very first step, make sure you're logged in into the HF Hub on all machines you're running your training on:

    hf auth login

We can then start with a few crucial imports:

In [19]:
import random
import time

import numpy as np
import trackio

import ray
from ray import tune
from ray.air.integrations.trackio import (
    TrackioLoggerCallback,
    setup_trackio,
)
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

PROJECT_NAME = "trackio-ray-example"

HF_DATASET_ID = "AINovice2005/ray-trackio-experiments"
HF_SPACE_ID = "AINovice2005/ray-trackio-dashboard"

NUM_STEPS = 15

## Tracking Ray Tune Experiments with Trackio

This example demonstrates how to integrate Trackio with Ray Tune using `TrackioLoggerCallback`. The callback automatically captures:

- Metrics reported via `tune.report`
- GPU utilization and system telemetry
- Trial configuration and metadata

All trials are logged under a single project, enabling structured comparison across hyperparameter configurations. Results can be persisted to a dataset or space for analysis and sharing with the community.

In [20]:
def tune_trainable(config):

    for step in range(NUM_STEPS):

        loss = (config["lr"] * 10) / (step + 1) + random.random()
        accuracy = 1 / (loss + 1e-3)

        # Example artifact
        image = np.random.rand(64, 64, 3)

        tune.report(
            {
                "loss": loss,
                "accuracy": accuracy,
                "image_mean": float(image.mean()),
                "step": step,
            }
        )

        time.sleep(0.2)

In [21]:
def run_tune_example():

    tuner = tune.Tuner(
        tune_trainable,
        param_space={
            "lr": tune.grid_search([0.001, 0.01, 0.1]),
        },
        run_config=tune.RunConfig(
            name="trackio-ray-tune-demo",
            callbacks=[
                TrackioLoggerCallback(
                    project=PROJECT_NAME,
                    auto_log_gpu=True,
                    gpu_log_interval=5,
                    dataset_id=HF_DATASET_ID,
                    space_id=HF_SPACE_ID,
                )
            ],
        ),
    )

    results = tuner.fit()

    print("\nTune finished\n")
    print(results)

## Applying Ray Train with Trackio

This example demonstrates how to integrate Trackio with Ray Train using setup_trackio, enabling manual logging within a training loop.

Here, we simulate the training loop which logs the scalar metrics such as loss and throughput
Step-wise progress for time-series tracking



In [22]:
def train_loop(config):

    run = setup_trackio(
        config=config,
        project=PROJECT_NAME,
        auto_log_gpu=True,
        gpu_log_interval=5,
        dataset_id=HF_DATASET_ID,
        space_id=HF_SPACE_ID,
    )

    for step in range(NUM_STEPS):

        loss = 5 / (step + 1) + random.random()
        throughput = random.uniform(50, 150)

        if run:
            run.log(
                {
                    "loss": loss,
                    "throughput": throughput,
                    "step": step,
                },
                step=step,
            )

        sample_image = np.random.rand(64, 64, 3)

        if run:
            run.log(
                {
                    "image_mean": float(sample_image.mean()),
                    "image_std": float(sample_image.std()),
                }
            )

        time.sleep(0.2)

    if run:
        run.finish()

In [23]:
def run_train_example():

    trainer = TorchTrainer(
        train_loop_per_worker=train_loop,
        train_loop_config={"lr": 0.01},
        scaling_config=ScalingConfig(num_workers=1),
        run_config=RunConfig(name="trackio-ray-train-demo"),
    )

    trainer.fit()

## Execute the example



In [24]:
ray.init(ignore_reinit_error=True)

print("\nRunning Ray Tune experiment\n")
run_tune_example()

print("\nRunning Ray Train experiment\n")
run_train_example()

print("\nOpening dashboard\n")
print("Run manually if needed:")
print('trackio show --project "trackio-ray-demo"')

trackio.finish()
ray.shutdown()

print("\nExecution completed\n")

* Trackio project initialized: trackio-ray-example
* Trackio metrics will be synced to Hugging Face Dataset: AINovice2005/ray-trackio-experiments
* Found existing space: https://huggingface.co/spaces/AINovice2005/ray-trackio-dashboard
* View dashboard by going to: https://AINovice2005-ray-trackio-dashboard.hf.space/


2026-03-28 06:33:06,305	WARNING trackio.py:286 -- trackio: Dropping unsupported metric 'checkpoint_dir_name' (type=NoneType). Only int/float supported.
2026-03-28 06:33:06,306	WARNING trackio.py:286 -- trackio: Dropping unsupported metric 'trial_id' (type=str). Only int/float supported.
2026-03-28 06:33:06,307	WARNING trackio.py:286 -- trackio: Dropping unsupported metric 'date' (type=str). Only int/float supported.
2026-03-28 06:33:06,308	WARNING trackio.py:286 -- trackio: Dropping unsupported metric 'hostname' (type=str). Only int/float supported.
2026-03-28 06:33:06,309	WARNING trackio.py:286 -- trackio: Dropping unsupported metric 'node_ip' (type=str). Only int/float supported.


* Created new run: tune_trainable_f8e3d_00002
* Created new run: tune_trainable_f8e3d_00001
* Created new run: tune_trainable_f8e3d_00000


/teamspace/studios/this_studio/ray/.venv/lib/python3.12/site-packages/trackio/run.py:608: UserWarning: Reserved keys renamed: ['step', 'timestamp'] → '__{key}'
  warnings.warn(f"Reserved keys renamed: {renamed_keys} → '__{{key}}'")
/teamspace/studios/this_studio/ray/.venv/lib/python3.12/site-packages/trackio/run.py:735: UserWarning: * Some logs could not be sent to the Space (it may still be starting up). They have been saved locally and will be sent automatically next time you call: trackio.init(project="trackio-ray-example", space_id="AINovice2005/ray-trackio-dashboard")
  warnings.warn(
2026-03-28 06:33:09,455	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/teamspace/studios/this_studio/ray_results/trackio-ray-tune-demo' in 0.0169s.


* Run finished. Uploading logs to Trackio Space (please wait...)
* Run finished. Uploading logs to Trackio Space (please wait...)
* Run finished. Uploading logs to Trackio Space (please wait...)


2026-03-28 06:33:09,472	INFO tune.py:1033 -- Total run time: 6.42 seconds (6.37 seconds for the tuning loop).


* Run finished. Uploading logs to Trackio Space (please wait...)

Tune finished

ResultGrid<[
  Result(
    metrics={'loss': 0.23413880737172652, 'accuracy': 4.252807144756496, 'image_mean': 0.4993654140354737, 'step': 14},
    path='/teamspace/studios/this_studio/ray_results/trackio-ray-tune-demo/tune_trainable_f8e3d_00000_0_lr=0.0010_2026-03-28_06-33-03',
    filesystem='local',
    checkpoint=None
  ),
  Result(
    metrics={'loss': 0.05223626356840479, 'accuracy': 18.784188313950164, 'image_mean': 0.4993381312907767, 'step': 14},
    path='/teamspace/studios/this_studio/ray_results/trackio-ray-tune-demo/tune_trainable_f8e3d_00001_1_lr=0.0100_2026-03-28_06-33-03',
    filesystem='local',
    checkpoint=None
  ),
  Result(
    metrics={'loss': 0.39510361260059845, 'accuracy': 2.52459197085972, 'image_mean': 0.5001492248008005, 'step': 14},
    path='/teamspace/studios/this_studio/ray_results/trackio-ray-tune-demo/tune_trainable_f8e3d_00002_2_lr=0.1000_2026-03-28_06-33-03',
    filesy

(TrainController pid=163866) Requesting resources: {'CPU': 1} * 1
(TrainController pid=163866) Attempting to start training worker group of size 1 with the following resources: [{'CPU': 1}] * 1
(RayTrainWorker pid=164541) Setting up process group for: env:// [rank=0, world_size=1]
(TrainController pid=163866) Started training worker group of size 1: 
(TrainController pid=163866) - (ip=10.128.0.146, pid=164541) world_rank=0, local_rank=0, node_rank=0
(RayTrainWorker pid=164541) * Trackio project initialized: trackio-ray-example
(RayTrainWorker pid=164541) * Trackio metrics will be synced to Hugging Face Dataset: AINovice2005/ray-trackio-experiments
(RayTrainWorker pid=164541) * Found existing space: https://huggingface.co/spaces/AINovice2005/ray-trackio-dashboard
(RayTrainWorker pid=164541) * View dashboard by going to: https://AINovice2005-ray-trackio-dashboard.hf.space/
(RayTrainWorker pid=164541) * Created new run: AINovice2005-1774679604
(RayTrainWorker pid=164541) /teamspace/studio


Opening dashboard

Run manually if needed:
trackio show --project "trackio-ray-demo"
* Run finished. Uploading logs to Trackio Space (please wait...)

Execution completed

